# Apigee Template: REST-AI-Images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-template-repository/blob/main/notebooks/REST-AI-Images.ipynb)

**Template Name:** `REST-AI-Images`  
**Description:** Unified OpenAI-compatible image generation API proxy (`/v1/images/generations`) providing image synthesis across Google Cloud Vertex AI Imagen (e.g. `imagen-3.0-generate-002`) and OpenAI (`dall-e-3`, `dall-e-2`) with automated payload transformation, IAM token generation, and response normalization.

### Key Capabilities:
- **OpenAI-Compatible `/v1/images/generations` Protocol:** Accept standard OpenAI image generation requests (`{"prompt": "...", "n": 1, "response_format": "b64_json"}`).
- **Protocol & Format Mediation:** Translates OpenAI image parameters into Google Cloud Vertex AI Imagen instances and parameters (`:predict`), converting returned base64 bytes or GCS URIs back to the standard OpenAI image schema.
- **Multi-Provider Routing:** Routes requests dynamically to Google Cloud Vertex AI or OpenAI based on the requested model.
- **Automated IAM & Security:** Automatically attaches Google Cloud OAuth tokens via `AM-SetGoogleToken`, removing the need for client apps to manage GCP credentials directly.
- **In-Notebook Image Rendering:** Decodes and renders generated images directly within the notebook.

### Documentation & References:
- [OpenAI Image Generations API Reference](https://platform.openai.com/docs/api-reference/images/create)
- [Google Cloud Vertex AI Imagen 3 Documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/image/generate-images)
- [Apigee Feature Templater (aft) GitHub](https://github.com/apigee/apigee-templater)
- [Apigee Data Collectors Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/data-collectors)

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID (Apigee Organization), Apigee environment, and optional OpenAI API key.

In [ ]:
# @title 1. Configuration & Authentication
import os

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
GOOGLE_CLOUD_PROJECT = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
OPENAI_API_KEY = ""  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ORG"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ENV"] = APIGEE_ENV
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["APIGEE_SA"] = f"apigee-service@{GOOGLE_CLOUD_PROJECT}.iam.gserviceaccount.com"

try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Authenticated with Google Cloud for project: {GOOGLE_CLOUD_PROJECT}")
except ImportError:
    print("Running outside Google Colab.")


## 2. Setup Tools, Initialize Resources & Deploy Template

Installs `aft`, runs `sh/initialize.sh` (configures service account, IAM bindings, data collectors, and custom reports), resolves `APIGEE_HOST`, and deploys `REST-AI-Images.yaml`.

In [ ]:
# @title 2. Setup, Initialize & Deploy Template
import os
import subprocess

# 1. Install Apigee Feature Templater (aft) CLI if needed
!which aft >/dev/null 2>&1 || curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 2. Download template and initialize script if running standalone
REPO_RAW = "https://raw.githubusercontent.com/gcp-samples/apigee-template-repository/main"
TEMPLATE_FILE = "REST-AI-Images.yaml" if os.path.exists("REST-AI-Images.yaml") else "templates/REST-AI-Images.yaml" if os.path.exists("templates/REST-AI-Images.yaml") else "REST-AI-Images.yaml"
os.environ["TEMPLATE_FILE"] = TEMPLATE_FILE

if not os.path.exists(TEMPLATE_FILE):
    !curl -fsSL -O {REPO_RAW}/templates/REST-AI-Images.yaml

if not os.path.exists("sh/initialize.sh"):
    !mkdir -p sh && curl -fsSL -o sh/initialize.sh {REPO_RAW}/sh/initialize.sh

# 3. Run initialization script
!bash sh/initialize.sh

# 4. Resolve APIGEE_HOST using aft describe
cmd = 'aft describe --project "$GOOGLE_CLOUD_PROJECT" -f json | jq --raw-output ".environmentGroups[] | select(any(.attachments[]; .environment == \"$APIGEE_ENV\")) | .hostnames[0]"'
try:
    host = subprocess.check_output(cmd, shell=True, text=True).strip()
    if host and host != "null":
        os.environ["APIGEE_HOST"] = host
        print(f"APIGEE_HOST resolved to: {host}")
except Exception as e:
    print(f"Could not automatically resolve APIGEE_HOST via aft: {e}")

# 5. Deploy template with aft
!aft "$TEMPLATE_FILE" \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --env="$APIGEE_ENV" \
  --sa="$APIGEE_SA"


## 3. Test Image Generation API via APIGEE_HOST

Send requests to `https://${APIGEE_HOST}/v1/images/generations` using standard OpenAI image generation requests.

In [ ]:
# @title Setup Test Client & APIGEE_HOST
import os
import base64
import json
import requests
import subprocess
from IPython.display import Image, display

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "")
APIGEE_ENV = os.getenv("APIGEE_ENV", "dev")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Determine APIGEE_HOST
APIGEE_HOST = os.getenv("APIGEE_HOST")
if not APIGEE_HOST or APIGEE_HOST == "null":
    APIGEE_HOST = f"{PROJECT_ID}-{APIGEE_ENV}.apigee.net"
    os.environ["APIGEE_HOST"] = APIGEE_HOST

# Retrieve caller access token if available
try:
    gcp_token = subprocess.check_output(["gcloud", "auth", "application-default", "print-access-token"], text=True).strip()
except Exception:
    try:
        gcp_token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()
    except Exception:
        gcp_token = ""

print(f"APIGEE_HOST: {APIGEE_HOST}")
print(f"Images Endpoint: https://{APIGEE_HOST}/v1/images/generations")

def generate_image(prompt: str, model: str = "imagen-3.0-generate-002", n: int = 1, response_format: str = "b64_json"):
    """
    Calls the Apigee /v1/images/generations endpoint with an OpenAI-compatible payload.
    Displays the generated image if returned as b64_json or url.
    """
    url = f"https://{APIGEE_HOST}/v1/images/generations"
    headers = {"Content-Type": "application/json"}
    if gcp_token:
        headers["Authorization"] = f"Bearer {gcp_token}"
    if OPENAI_API_KEY:
        headers["x-api-key"] = OPENAI_API_KEY

    payload = {
        "model": model,
        "prompt": prompt,
        "n": n,
        "response_format": response_format
    }

    print(f"\n---> Sending [{model}] image generation request for prompt:\n     \"{prompt}\"")
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        print(f"HTTP Status: {response.status_code}")
        try:
            data = response.json()
            images = data.get("data", [])
            print(f"Generated Images Count: {len(images)}")
            for idx, img in enumerate(images):
                if "b64_json" in img:
                    img_bytes = base64.b64decode(img["b64_json"])
                    print(f"Image #{idx+1} rendered ({len(img_bytes)} bytes):")
                    display(Image(data=img_bytes))
                elif "url" in img:
                    print(f"Image #{idx+1} URL: {img['url']}")
                    display(Image(url=img["url"]))
            return data
        except Exception:
            print(response.text)
            return None
    except Exception as e:
        print("Request error:", e)
        return None


In [ ]:
# @title Test 1: Generate Image with Google Cloud Imagen 3
# Apigee translates the OpenAI request to Vertex AI Imagen format and normalizes the response
res1 = generate_image(
    prompt="A futuristic smart city with lush vertical gardens, flying autonomous transport, and glowing solar glass towers at twilight, photorealistic 8k",
    model="imagen-3.0-generate-002",
    n=1,
    response_format="b64_json"
)


In [ ]:
# @title Test 2: Generate Image with OpenAI DALL-E (Optional)
# Requires OPENAI_API_KEY to be configured; Apigee forwards to OpenAI target
if OPENAI_API_KEY:
    res2 = generate_image(
        prompt="Minimalist origami bird flying above a calm mountain lake at golden hour, watercolor art style",
        model="dall-e-3",
        n=1,
        response_format="b64_json"
    )
else:
    print("Skipping OpenAI DALL-E test: OPENAI_API_KEY is not set.")


## 4. Verify Apigee Analytics & Token Data Collection

Image generation calls are governed and audited through Apigee:
- **Target Routing:** Requests routed to Google Cloud Vertex AI use automatic IAM OAuth bearer minting (`AM-SetGoogleToken`).
- **Observability:** Audit logs and transaction metrics are tracked under **Analytics > Custom Reports** in the [Apigee Console](https://console.cloud.google.com/apigee).